Entry Condition: RSI(14) > 70

Exit (TP): Price >= EntryPrice * 1.06

Exit (SL): RSI(14) < 55

# Python Code

In [ ]:
# Install required libraries
!pip install pandas pandas-ta requests plotly -q

import requests
import time
import pandas as pd
import pandas_ta as ta
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

# ==========================================
# GLOBAL CONFIGURATION
# ==========================================
CONFIG = {
    'SYMBOLS': ['BTCUSDT', 'ETHUSDT'],
    'INTERVAL': '4h',                   # Khung thời gian 4H
    'START_DATE': '01/01/2020',         # Dữ liệu từ 2020
    'END_DATE': '01/01/2022',
    'INITIAL_CAPITAL': 10000.0,
    'POSITION_SIZE_PCT': 0.10,          # 10% vốn mỗi lệnh
    'FEE_RATE': 0.0004,
    'SLIPPAGE': 0.0002
}

# ==========================================
# MODULE 1: DATA HANDLER
# ==========================================
class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.symbol = symbol.upper()
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return int(dt.timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"
        req_symbol = self.symbol

        print(f"Checking connection for {req_symbol}...")
        try:
            test = requests.get(f"{base_url}/api/v3/time", timeout=5)
            if test.status_code == 451:
                base_url = "https://api.binance.us"
        except Exception:
            base_url = "https://api.binance.us"

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)

        if self.end_date == datetime.now().strftime("%d/%m/%Y"):
            end_time = int(time.time() * 1000)
        else:
            end_dt = datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
            end_time = int(end_dt.timestamp() * 1000)

        all_klines =[]
        print(f"Downloading {req_symbol} ({self.interval})...")

        while start_time < end_time:
            params = {"symbol": req_symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except Exception: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        df = df.astype(float)
        print(f"Total: {len(df)} candles fetched.")
        return df

# ==========================================
# MODULE 2: STRATEGY LOGIC (RSI MOMENTUM)
# ==========================================
class MomentumRsiStrategy:
    def __init__(self):
        self.rsi_period = 14
        self.rsi_entry = 70
        self.rsi_sl = 55
        self.tp_pct = 0.06  # Take Profit 6%

    def calculate_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        df['RSI'] = ta.rsi(df['Close'], length=self.rsi_period)
        df.dropna(inplace=True)
        return df

# ==========================================
# MODULE 3: BACKTEST ENGINE
# ==========================================
class BacktestEngine:
    def __init__(self, df, strategy, config):
        self.df = strategy.calculate_indicators(df.copy())
        self.strategy = strategy
        self.config = config

        self.capital = config['INITIAL_CAPITAL']
        self.invested_amount = 0.0
        self.position = 0
        self.entry_price = 0.0
        self.trades =[]

        self.equity_curve = np.zeros(len(self.df))

    def run(self):
        O = self.df['Open'].values
        H = self.df['High'].values
        C = self.df['Close'].values
        times = self.df.index
        rsi = self.df['RSI'].values

        fee = self.config['FEE_RATE']
        slip = self.config['SLIPPAGE']
        pos_size_pct = self.config['POSITION_SIZE_PCT']

        s = self.strategy

        self.equity_curve[0] = self.capital

        for i in range(1, len(self.df)):
            # ---------------------------
            # EXIT LOGIC
            # ---------------------------
            if self.position == 1:
                exit_signal = False
                reason = ""
                real_exit = 0.0

                # Tính giá Take Profit
                tp_price = self.entry_price * (1 + s.tp_pct)

                # 1. Take Profit Logic (Intraday Limit Order)
                # Chốt lời nếu giá High chạm hoặc vượt mốc +6%
                if H[i] >= tp_price:
                    exit_signal = True
                    reason = "TP (+6%)"
                    # Nếu mở phiên nhảy Gap up lớn hơn TP, thì chốt ở Open. Nếu không, chốt đúng giá TP.
                    real_exit = max(O[i], tp_price) * (1 - slip)

                # 2. Stop Loss Logic (RSI < 55)
                # Kiểm tra RSI của nến TRƯỚC ĐÓ đã đóng dưới 55 chưa
                elif rsi[i-1] < s.rsi_sl:
                    exit_signal = True
                    reason = "SL (RSI < 55)"
                    real_exit = O[i] * (1 - slip)

                # Thực thi Exit
                if exit_signal:
                    pnl_pct = (real_exit - self.entry_price) / self.entry_price
                    net_pnl_pct = pnl_pct - (fee * 2)
                    pnl_amt = self.invested_amount * net_pnl_pct
                    self.capital += pnl_amt

                    self.trades.append({
                        'Date': times[i], 'Type': 'Long',
                        'Reason': reason,
                        'Entry': self.entry_price, 'Exit': real_exit,
                        'PnL_Amt': pnl_amt, 'Equity': self.capital
                    })

                    self.position = 0
                    self.invested_amount = 0.0

            # ---------------------------
            # ENTRY LOGIC
            # ---------------------------
            if self.position == 0:
                # Điều kiện: RSI nến trước ĐÓNG CỬA > 70
                if rsi[i-1] > s.rsi_entry:
                    self.position = 1
                    self.entry_price = O[i] * (1 + slip)
                    self.invested_amount = self.capital * pos_size_pct

            # ---------------------------
            # CALCULATE UNREALIZED PNL
            # ---------------------------
            current_equity = self.capital
            if self.position == 1:
                current_value = self.invested_amount * (C[i] / self.entry_price)
                unrealized_pnl = current_value - self.invested_amount
                current_equity += unrealized_pnl

            self.equity_curve[i] = current_equity

        return pd.Series(self.equity_curve, index=self.df.index), pd.DataFrame(self.trades)

# ==========================================
# MODULE 4: PERFORMANCE EVALUATOR
# ==========================================
class Evaluator:
    @staticmethod
    def generate_report(equity, trades_df, initial_capital, symbol, interval, pos_size_pct):
        if trades_df.empty:
            print(f"No trades executed for {symbol}!")
            return

        final_equity = equity.iloc[-1]
        net_pnl = final_equity - initial_capital
        total_return = net_pnl / initial_capital

        rolling_max = equity.cummax()
        drawdown = (equity - rolling_max) / rolling_max
        max_dd = drawdown.min()

        wins = trades_df[trades_df['PnL_Amt'] > 0]
        losses = trades_df[trades_df['PnL_Amt'] <= 0]
        winrate = len(wins) / len(trades_df) if len(trades_df) > 0 else 0
        profit_factor = wins['PnL_Amt'].sum() / abs(losses['PnL_Amt'].sum()) if len(losses) > 0 and losses['PnL_Amt'].sum() != 0 else 0

        days = (equity.index[-1] - equity.index[0]).days
        cagr = ((final_equity / initial_capital) ** (365.0 / days)) - 1 if days > 0 else 0

        # Sharpe ratio (4H có 6 nến 1 ngày -> 365 * 6)
        returns = equity.pct_change().dropna()
        sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(365 * 6) if returns.std() != 0 else 0

        print("")
        print(f"========== PERFORMANCE REPORT: {symbol} ({interval}) ==========")
        print(f"{'Strategy:':>20} RSI Momentum (>70)")
        print(f"{'Position Sizing:':>20} {pos_size_pct*100}% of Capital")
        print(f"{'Initial Capital:':>20} ${initial_capital:,.2f}")
        print(f"{'Final Equity:':>20} ${final_equity:,.2f}")
        print(f"{'Net PnL:':>20} ${net_pnl:,.2f}")
        print(f"{'Total Return:':>20} {total_return:.2%}")
        print(f"{'CAGR (Annual):':>20} {cagr:.2%}")
        print(f"{'Max Drawdown:':>20} {max_dd:.2%}")
        print(f"{'Sharpe Ratio:':>20} {sharpe_ratio:.2f}")
        print(f"{'Total Trades:':>20} {len(trades_df)}")
        print(f"{'Winrate:':>20} {winrate:.2%}")
        print(f"{'Profit Factor:':>20} {profit_factor:.2f}")
        print("====================================================\n")

        # Plotly Charts
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.7, 0.3])
        fig.add_trace(go.Scatter(x=equity.index, y=equity.values, mode="lines", name="Equity", line=dict(color="#2962ff", width=1.5)), row=1, col=1)
        fig.add_trace(go.Scatter(x=drawdown.index, y=drawdown.values, mode="lines", name="Drawdown", line=dict(color="#d32f2f", width=1), fill='tozeroy', fillcolor='rgba(211, 47, 47, 0.2)'), row=2, col=1)

        fig.update_layout(title=f"Portfolio Performance: {symbol} ({interval})", template="plotly_white", height=650, margin=dict(l=40, r=40, t=60, b=40), showlegend=False)
        fig.update_xaxes(showgrid=True, gridcolor='#e6e6e6', zeroline=False)
        fig.update_yaxes(title_text="Balance ($)", tickprefix="$", showgrid=True, gridcolor='#e6e6e6', zeroline=False, row=1, col=1)
        fig.update_yaxes(title_text="Drawdown %", tickformat=".1%", showgrid=True, gridcolor='#e6e6e6', zeroline=True, zerolinecolor='#cccccc', row=2, col=1)

        fig.show()

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    for symbol in CONFIG['SYMBOLS']:
        print(f"\n---> Starting Backtest for: {symbol} <---")
        fetcher = BinanceDataFetcher(symbol, CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])
        df_raw = fetcher.fetch()

        if not df_raw.empty:
            my_strategy = MomentumRsiStrategy()
            engine = BacktestEngine(df_raw, my_strategy, CONFIG)
            equity, trades = engine.run()

            Evaluator.generate_report(
                equity,
                trades,
                CONFIG['INITIAL_CAPITAL'],
                symbol,
                CONFIG['INTERVAL'],
                CONFIG['POSITION_SIZE_PCT']
            )

            # Print recent trades to verify logic
            if not trades.empty:
                print(f"\nTop 5 Recent Trades for {symbol}:")
                print(trades[['Date', 'Type', 'Reason', 'Entry', 'Exit', 'PnL_Amt']].tail(5).to_string(index=False))


---> Starting Backtest for: BTCUSDT <---
Checking connection for BTCUSDT...
Total: 4391 candles fetched.

========== PERFORMANCE REPORT: BTCUSDT (4h) ==========
           Strategy: RSI Momentum (>70)
    Position Sizing: 10.0% of Capital
    Initial Capital: $10,000.00
       Final Equity: $11,698.23
            Net PnL: $1,698.23
       Total Return: 16.98%
      CAGR (Annual): 8.15%
       Max Drawdown: -2.27%
       Sharpe Ratio: 2.14
       Total Trades: 88
            Winrate: 57.95%
      Profit Factor: 2.29




Top 5 Recent Trades for BTCUSDT:
               Date Type        Reason        Entry         Exit    PnL_Amt
2021-10-12 20:00:00 Long SL (RSI < 55) 54995.196840 55318.714044   5.964384
2021-10-20 12:00:00 Long      TP (+6%) 61458.619266 65133.107195  69.256444
2021-10-21 16:00:00 Long SL (RSI < 55) 66889.215168 62958.605760 -70.344023
2021-11-11 00:00:00 Long SL (RSI < 55) 65213.040000 64869.443516  -7.124614
2021-12-26 04:00:00 Long SL (RSI < 55) 51120.222000 49665.624888 -34.322932

---> Starting Backtest for: ETHUSDT <---
Checking connection for ETHUSDT...
Total: 4391 candles fetched.

========== PERFORMANCE REPORT: ETHUSDT (4h) ==========
           Strategy: RSI Momentum (>70)
    Position Sizing: 10.0% of Capital
    Initial Capital: $10,000.00
       Final Equity: $13,297.31
            Net PnL: $3,297.31
       Total Return: 32.97%
      CAGR (Annual): 15.29%
       Max Drawdown: -3.01%
       Sharpe Ratio: 2.89
       Total Trades: 123
            Winrate: 65.04%
      Profit


Top 5 Recent Trades for ETHUSDT:
               Date Type        Reason       Entry        Exit    PnL_Amt
2021-10-21 08:00:00 Long      TP (+6%) 4081.816200 4325.859827  78.825791
2021-10-22 16:00:00 Long SL (RSI < 55) 4163.372508 3965.736694 -64.883965
2021-11-04 16:00:00 Long SL (RSI < 55) 4611.542124 4479.433934 -39.391533
2021-11-10 04:00:00 Long SL (RSI < 55) 4721.454102 4690.541704  -9.799407
2021-12-02 04:00:00 Long SL (RSI < 55) 4591.388094 4490.121796 -30.461562


# Go Code